# Spin Orbite model for 3d orbital
## Author: Mathieu Desmarais
### Date: 28-05-2026 

## Packages and modules

In [2]:
import numpy as np 
import qutip as qt                 # Package for quantum mecanic
import ufss as uf                  # Generation of doubled sided feynmann diagram
import matplotlib.pyplot as plt
from itertools import product
from unittest.mock import patch
import os

# 2. La spectroscopie 2D
from qudpy.Classes import System   # Calcul and generation of 2D plot
import qudpy.plot_functions as pf


## Feynmann Diagram

In [3]:
DG = uf.DiagramGenerator
R3rd = DG()
dummy_times = [0, 100, 200, 300]
t_pulse = np.array([-1, 1])
R3rd.efield_times = [t_pulse] * 4

# --- 1. Diagrammes Rephasing (R1, R2, R3) ---
# Phase discrimination pour k_I = -k1 + k2 + k3
R3rd.set_phase_discrimination([(0, 1), (1, 0), (1, 0)])
[R3, R1, R2] = R3rd.get_diagrams(dummy_times)
rephasing = [R1, R2]

# --- 2. Diagrammes Non-Rephasing (R4, R5, R6) ---
# Phase discrimination pour k_II = +k1 - k2 + k3
R3rd.set_phase_discrimination([(1, 0), (0, 1), (1, 0)])
[R6,R4, R5] = R3rd.get_diagrams(dummy_times)
nonrephasing = [R5,R6]
response_list = []
total_diagrams = rephasing + nonrephasing

## Constante

In [4]:



hbar = 0.658211951 # hbar in eV*fs

## Useful Functions


In [5]:
#### Calcul of the dispersion for the spin-Peierls phases #####
def Spin_Peierls_dispersion(k, T, B, J, T_SP_0, delta_0, beta):
    alpha = 0.004
    T_SP = T_SP_0 * (1 - alpha * B**2)
    T_SP = max(0.0,T_SP)

    if T < T_SP: 
        delta = delta_0 * (1 - T/T_SP)**beta
    else: 
        delta=0

    #Cross-Fisher relation: the gap Delta scale like J* delta^(2/3)
    Delta = 2.0 * J * (delta **(2/3)) if delta > 0 else 0.0

    v = (np.pi * J) /2
    epsilon_k = np.sqrt(Delta**2 + ((v * np.sin(k))**2))
    return epsilon_k, Delta

def get_coupling(k, delta_dimerisation, alpha_dimerisation, a_dimer=1.0, parity='odd'):
    """
    Calculates the effective SOC coupling for a dimerized lattice (e.g., CuGeO3).

    Parameters:
    -----------
    k : float
        Wave vector.
    eta : float
        Mixing parameter. Represents symmetry breaking due to lattice 
        fluctuations, magnetic frustration, or crystal field effects.
    a_dimer : float
        Relative distance between atoms in the dimer (normalized).
    parity : str
        'odd' (antisymmetric -> sin), 'even' (symmetric -> cos), or 'mixed'.

    Returns:
    --------
    float
        The effective coupling strength at wave vector k.
    """
    eta = delta_dimerisation * alpha_dimerisation # Proportional to the dimerization strength, modulated by a small factor alpha
    # The phase shift depends on the internal dimension of the dimer
    phase = k * (a_dimer / 2.0)
    
    if parity == 'odd':
        return np.sin(phase)
    elif parity == 'even':
        return np.cos(phase)
    else:
        # Arbitrary mixing if the crystal symmetry is broken
        return np.sin(phase) + eta * np.cos(phase)
    
def get_subspace_projector(dims, condition_func):
    """
    Construit le projecteur P pour un espace défini par 'dims'.
    condition_func(indices) doit retourner True si l'état est valide.
    """
    basis_subset = []
    
    # Génère toutes les combinaisons d'indices possibles pour la base complète
    # ex: pour dims=[3, 2, 2], product génère (0,0,0), (0,0,1), ..., (2,1,1)
    all_indices = product(*[range(d) for d in dims])
    
    for indices in all_indices:
        # Appliquer votre condition logique
        # 'indices' est un tuple, ex: (orb_idx, spin1_idx, spin2_idx, ...)
        if condition_func(indices):
            # Créer le vecteur de base complet correspondant
            ket = qt.tensor([qt.basis(d, i) for d, i in zip(dims, indices)])
            basis_subset.append(ket.full()) # On garde la forme numpy pour l'instant
    
    # Empiler les vecteurs colonnes pour former P (N_full x N_sub)
    if not basis_subset:
        raise ValueError("Aucun état ne satisfait la condition.")
        
    P_data = np.hstack(basis_subset)
    return qt.Qobj(P_data, dims=[dims, [len(basis_subset)]])

def mon_critere_physique(indices):
    excitations_spin = sum(indices[1:]) # Somme des indices des modes
    return excitations_spin <= 1

def applyProjector(P,H_tot,c_ops_tot, lowering_operator, dipole_operator):
    H_tot_reduced =P.dag() *H_tot * P
    c_ops_reduced = [P.dag() * c_op * P for c_op in c_ops_tot]
    a_reduced = P.dag() * lowering_operator * P
    dipole_operator_reduced = P.dag() * dipole_operator *P
    return H_tot_reduced, c_ops_reduced, a_reduced, dipole_operator_reduced

def build_collapse_operators(ket_g, ket_b, annihilations_ops, gamma_orb, gamma_spin,gamma_orb_self, dims_orb, dims_spin):
    """
    Construit la liste des opérateurs de Lindblad (c_ops) pour le système couplé.
    dims_orb: Liste des dimensions orbitales (ex: [3])
    dims_spin: Liste des dimensions de spin (ex: [2, 2] pour 2 modes)
    """
    c_ops = []
    
    # 0. Création des opérateurs Identité avec la bonne structure tensorielle
    # qt.qeye([2, 2]) crée automatiquement la structure dims=[[2,2], [2,2]]
    I_spin = qt.qeye(dims_spin)
    I_orb = qt.qeye(dims_orb)
    
    # --- 1. Orbital Relaxation ---
    # L'électron descend de Bright à Ground
    decay_orb = ket_g * ket_b.dag()
    # self orbital decay operator on the bright state (for population loss without going to ground)
    decay_orb_self = ket_b * ket_b.dag()
    
    # Le produit tensoriel assemble [3] et [2, 2] en [3, 2, 2] naturellement
    c_orb_tot = qt.tensor(decay_orb, I_spin)
    c_orb_self_tot = qt.tensor(decay_orb_self, I_spin)

    c_ops.append(np.sqrt(gamma_orb) * c_orb_tot)
    c_ops.append(np.sqrt(gamma_orb_self) * c_orb_self_tot)

    # --- 2. Triplon Relaxation (3D effect / Decoherence) ---
    # Applique un taux de perte sur CHAQUE mode k
    for a_k in annihilations_ops:
        # a_k est déjà dans l'espace de spin total (dims=[[2, 2], [2, 2]])
        c_spin_tot = qt.tensor(I_orb, a_k)
        c_ops.append(np.sqrt(gamma_spin) * c_spin_tot)
        
    return c_ops

def sauvegarder_silva_plot(i, type_spectre, dossier_sauvegarde=".", spectra_list=None, x_val=None, y_val=None, **kwargs):
    """
    Sauvegarde le graphique silva_plot dans un dossier spécifique avec un nom personnalisé.
    
    :param i: Indice de la boucle temporelle (int)
    :param type_spectre: Chaîne de caractères (ex: "rephasing" ou "nonrephasing")
    :param dossier_sauvegarde: Chemin du dossier où enregistrer l'image (str)
    """
    
    # 1. Sécurité : Créer le dossier s'il n'existe pas déjà
    os.makedirs(dossier_sauvegarde, exist_ok=True)
    
    # 2. Définir le nom du fichier (ex: rephasing_0001.png)
    nom_fichier_image = f"{type_spectre}_{i:04d}.png"
    
    # 3. Joindre le dossier et le nom du fichier de manière propre
    chemin_complet = os.path.join(dossier_sauvegarde, nom_fichier_image)
    
    # 4. Génération et sauvegarde de la figure
    with patch('matplotlib.pyplot.show'):
        pf.silva_plot(spectra_list=spectra_list, x_val=x_val, y_val=y_val, **kwargs)
        
        # On sauvegarde au chemin complet
        plt.savefig(chemin_complet, dpi=300, bbox_inches='tight')
        plt.close()
        
    print(f"Enregistré : {chemin_complet}")
######################################################################################
######################### Test functions #############################################
######################################################################################
def simulate_dynamics_soc(valeurs_lambda_soc, n_modes, H_orb, dipole_op, L_op, ket_g, ket_d, ket_b, H_spin, annihilations_ops, k_values, tlist, c_ops):
    print(f"--- Préparation des Hamiltoniens de base (N_modes = {n_modes}) ---")
    
 

    Id_spin = qt.qeye(H_spin.shape[0])
    Id_orb = qt.qeye(H_orb.shape[0])


    ket_0_spin = qt.basis(Id_spin.shape[0], 0)
    psi0 = qt.tensor(ket_b, ket_0_spin)

    P_g_tot = qt.tensor(ket_g * ket_g.dag(), Id_spin)
    P_d_tot = qt.tensor(ket_d * ket_d.dag(), Id_spin)
    P_b_tot = qt.tensor(ket_b * ket_b.dag(), Id_spin)

    resultats = {}

    print("--- Début de la boucle de simulation ---")
    for lamda in valeurs_lambda_soc:
        print(f"Simulation en cours pour lambda_SOC = {lamda}...")
        
        H_tot, dipole_tot, lowering_op_tot = TotalHamiltonian(
            H_orb, dipole_op, L_op, H_spin, annihilations_ops, k_values, Delta, lambda_SOC=lamda, a_dimer=1, parity="odd"
        )
        
        # --- LE CORRECTIF : Aplatissement de tous les opérateurs pour mesolve ---
        N_tot = H_tot.shape[0] # Vaudra 63 dans ton cas
        dims_op = [[N_tot], [N_tot]]
        dims_ket = [[N_tot], [1]]

        # 1. On uniformise l'Hamiltonien et l'état
        H_tot.dims = dims_op
        psi0.dims = dims_ket

        # 2. On uniformise les projecteurs
        P_g_tot.dims = dims_op
        P_d_tot.dims = dims_op
        P_b_tot.dims = dims_op

        # 3. On uniformise les opérateurs de Lindblad (c_ops)
        c_ops_flat = []
        for c in c_ops:
            c_copy = c.copy()
            c_copy.dims = dims_op
            c_ops_flat.append(c_copy)
        # ------------------------------------------------------------------------
        
        # Résolution de l'équation maîtresse avec les versions aplaties
        result = qt.mesolve(H_tot, psi0, tlist, c_ops=c_ops_flat, e_ops=[P_g_tot, P_d_tot, P_b_tot])
        
        pop_ground = result.expect[0]
        pop_dark = result.expect[1]
        pop_bright = result.expect[2]
        
        resultats[lamda] = {
            'ground': pop_ground,
            'dark': pop_dark,
            'bright': pop_bright
        }

        # Affichage Graphique
        plt.figure(figsize=(9, 5))
        plt.plot(tlist, pop_bright, label="Population (Bright State)", color="red", linewidth=2)
        plt.plot(tlist, pop_dark, label="Population (Dark State + Triplons)", color="blue", linewidth=2)
        plt.plot(tlist, pop_ground, label="Population (Ground State)", color="green", linewidth=2)
        plt.plot(tlist, pop_bright + pop_dark + pop_ground, label="Population Totale (Conservation)", color="black", linestyle="--")

        plt.xlabel("Temps")
        plt.ylabel("Probabilité")
        plt.title(f"Dynamique de transfert SOC ($\\lambda_{{SOC}}$ = {lamda}, $N_{{modes}}$ = {n_modes})")
        plt.legend()
        plt.grid(True)
        plt.show()
        
    print("--- Simulations terminées ---")
    return resultats


## Hamiltonian Definition

In [6]:
def BuildOrbitalHamiltonian(Delta_dark, Delta_bright):
    """
    Construction of the orbital sector (ground, Bright, Dark).
    """
    ket_g = qt.basis(3, 0) # |ground>
    ket_d = qt.basis(3, 1) # |dark>
    ket_b = qt.basis(3, 2) # |bright>

    # Energy projection operators
    P_d = ket_d * ket_d.dag()
    P_b = ket_b * ket_b.dag()  
    P_g = ket_g * ket_g.dag()  

    # Orbital Hamiltonian (sets energy levels for excited states)
    H_orb = Delta_dark * P_d + Delta_bright * P_b
    a_b_local = ket_g * ket_b.dag() + ket_b * ket_g.dag() # Local coupling between g and b 
    a_d_local = ket_g * ket_d.dag() + ket_d * ket_g.dag()

    # Dipolar Operator (Light-matter coupling: pump excites g -> b)
    dipole_op_bright = ket_g * ket_b.dag() + ket_b * ket_g.dag()
    dipole_op_dark = ket_g * ket_d.dag() + ket_d * ket_g.dag()

    # Orbital operator for Spin-Orbit Coupling (SOC) (Mixes b <-> d)
    L_op = ket_d * ket_b.dag() + ket_b * ket_d.dag() 

    return H_orb, dipole_op_bright, dipole_op_dark, L_op, ket_g, ket_d, ket_b

def BuildKSpaceSpinHamiltonian(N_modes, T, B, J, T_SP_0, delta_0, beta, max_bosons_per_mode=2, max_total_triplons=1):
    """
    Constructs the triplon sector in k-space.
    Uses 'enr_destroy' to restrict the Hilbert space size for efficiency.
    """
    # Brillouin Zone discretization
    k_values = np.linspace(-np.pi, np.pi, N_modes)
    results = [Spin_Peierls_dispersion(k, T, B, J, T_SP_0, delta_0, beta) for k in k_values]

    # 2. Split the dispersion relation and the dimerisation parameter
    epsilon_k = [res[0] for res in results]
    Delta_k = [res[1] for res in results]

# 3. Delta is cst in k, so we can take the first value (or any value since they should be the same)
    Delta = Delta_k[0]
    
    # Dimensions defined by max bosons allowed per mode
    dims = [max_bosons_per_mode] * N_modes
    
    # Energy-Restricted (ENR) annihilation operators
    # If max_total_triplons = 1, the matrix size is linear (N_modes + 1) instead of 2^N
    
    annihilation_ops = []
    H_spin = 0
    for i in range(N_modes): 
        
        op_list = [qt.qeye(max_bosons_per_mode) for _ in range (N_modes)]
        op_list[i] = qt.destroy(max_bosons_per_mode)
        annihilation_ops.append(qt.tensor(op_list))
        a_global = annihilation_ops[i]
        H_spin += epsilon_k[i] * a_global.dag() * a_global


    


    return H_spin, annihilation_ops, k_values, Delta

def TotalHamiltonian(H_orb, dipole_op, L_op, H_spin_k, annihilations_ops, k_values, Delta, Threshold=1e-6, 
                     lambda_SOC=0.01, alpha=0, a_dimer=1.0, parity='odd'):
    
    Id_orb = qt.qeye(H_orb.dims[0])
    Id_spin = qt.qeye(H_spin_k.dims[0])
    #Id_spin = qt.Qobj(np.eye(H_spin_k.shape[0]), dims=H_spin_k.dims)

    H_orb_tot = qt.tensor(H_orb, Id_spin)
    H_spin_tot = qt.tensor(Id_orb, H_spin_k)



    H_spin_tot.dims = H_orb_tot.dims

    H_SOC_tot = 0
    for i, k in enumerate(k_values):
        a_k = annihilations_ops[i]
        
        # Calling the coupling function
        V_k = get_coupling(k,Delta,alpha, a_dimer=a_dimer, parity=parity)
        
        S_op_k = V_k * (a_k + a_k.dag())
        H_SOC_term = qt.tensor(L_op, S_op_k)
        H_SOC_term.dims = H_orb_tot.dims
        H_SOC_tot += lambda_SOC * H_SOC_term

    # Combine and finalize



    
    H_tot = H_orb_tot + H_spin_tot + H_SOC_tot

    ket_g = qt.basis(3, 0) # |ground>
    ket_d = qt.basis(3, 1) # |dark>
    ket_b = qt.basis(3, 2) # |bright>

    dipole_operator = ket_g * ket_b.dag() + ket_b * ket_g.dag()
    dipole_operator = qt.tensor(dipole_operator, Id_spin)
    dipole_operator = dipole_operator.tidyup(atol=Threshold)

    lowering_op = ket_g * ket_b.dag()
    lowering_op = qt.tensor(lowering_op, Id_spin)
    lowering_op = lowering_op.tidyup(atol=Threshold)

    
    dipole_tot = qt.tensor(dipole_op, Id_spin)
   
    return H_tot, lowering_op, dipole_operator  



In [7]:
def simuler_et_sauvegarder(i, N_t, t2_attente, phys, plot_options,plot=0, dossier_1="Resultats_Spectres_rephasing",dossier_2="Resultats_Spectres_nonrephasing"):
    """
    Construit l'Hamiltonien, génère la matrice densité, calcule la dynamique 2D 
    et sauvegarde les spectres rephasing et non-rephasing.
    """
    print(f"\n--- [{i}] Préparation du système (N_t={N_t}, t2_attente={t2_attente}) ---")
    
    # 1. Création des Hamiltoniens
    H_orb, dipole_op_bright, dipole_op_dark, L_op, ket_g, ket_d, ket_b = BuildOrbitalHamiltonian(phys['E_d'], phys['E_B'])
    
    H_spin, annihilation_ops, k_values, Delta = BuildKSpaceSpinHamiltonian(
        phys['N_modes'], phys['T'], phys['B'], phys['J'], phys['T_SP_0'], 
        phys['delta_0'], phys['beta'], max_bosons_per_mode=phys['max_bosons_per_mode']
    )
    
    H_tot, lowering_operator, dipole_operator = TotalHamiltonian(
        H_orb, dipole_op_bright, L_op, H_spin, annihilation_ops, k_values, Delta, 
        alpha=0.01, lambda_SOC=phys['Lambda_SOC']
    )
    
    c_ops_tot = build_collapse_operators(
        ket_g, ket_b, annihilation_ops, phys['gamma_orb'], phys['gamma_spin'], 
        phys['gamma_orb_self'], H_orb.dims[0], H_spin.dims[0]
    )

    # 2. Projection et Initialisation du Système
    P = get_subspace_projector(H_tot.dims[0], mon_critere_physique)
    H_tot_reduced, c_ops_reduced, a_reduced, dipole_operator_reduced = applyProjector(
        P, H_tot, c_ops_tot, lowering_operator, dipole_operator
    )

    ket_fondamental = qt.basis(H_tot_reduced.shape[0], 0)
    rho_0 = qt.ket2dm(ket_fondamental)

    sys = System(H=H_tot_reduced, rho=rho_0, a=a_reduced, u=dipole_operator_reduced, 
                 c_ops=c_ops_reduced, diagonalize=False)

    # 3. Calcul de la Dynamique 2D
    time_delays = [N_t, t2_attente, N_t]
    scan_id = [0, 2]
    response_list = []
    
    print("Calcul de la dynamique quantique en cours...")
    for k in range(4):
        states, t1, t2, dipole = sys.coherence2d(time_delays, total_diagrams[k], scan_id, r=phys['res_fs'], parallel=False)
        response_list.append(1j * dipole)
        
    spectra_list, extent, f1, f2 = sys.spectra(response_list, resolution=phys['res_fs'])

    if plot == 0:
        rephasing_spectra = spectra_list[:2]
        rephasing_spectra.append(np.sum(spectra_list[:2], 0))
        sauvegarder_silva_plot(i, "rephasing", dossier_1, rephasing_spectra, f1, f2, **plot_options)

        # Non-Rephasing
        nonrephasing_spectra = spectra_list[2:]
        nonrephasing_spectra.append(np.sum(spectra_list[2:], 0))
        sauvegarder_silva_plot(i, "nonrephasing", dossier_2, nonrephasing_spectra, f1, f2, **plot_options)

    if plot == 1:
        rephasing_spectra = spectra_list[:2]
        rephasing_spectra.append(np.sum(spectra_list[:2], 0))
        sauvegarder_silva_plot(i, "rephasing", dossier_1, rephasing_spectra, f1, f2, **plot_options)
    if plot == 2:
        nonrephasing_spectra = spectra_list[2:]
        nonrephasing_spectra.append(np.sum(spectra_list[2:], 0))
        sauvegarder_silva_plot(i, "nonrephasing", dossier_2, nonrephasing_spectra, f1, f2, **plot_options)

In [ ]:
params_physiques = {
    'N_modes': 1,
    'T': 5,
    'T_SP_0': 14,
    'B': 0,
    'J': 1,
    'delta_0': 0.0,
    'beta': 0.5,
    'Lambda_SOC': 0.05,
    'max_bosons_per_mode': 2,
    'E_d': 0.90,
    'E_B': 1.0,
    'gamma_orb': 0.1,
    'gamma_spin': 0.1,
    'gamma_orb_self': 0.1,
    'res_fs': 1  # Résolution temporelle
}

params_plot = {
    'labels': ['E emission', 'E absorption'],
    'title_list': ['$R_5$', '$R_6$', '$R_{unrephasing}$'],
    'scale': 'linear',
    'color_map': 'jet',
    'interpolation': 'spline36',
    'center_scale': False,
    'plot_sum': False,
    'plot_quadrant': 'Zoom',
    'invert_y': False,
    'diagonals': [True, True],
    'Zoom_coor': [-2, 2, -2, 2]
}

N_steps_list = np.arange(10, 50,5 )
#N_steps_list = 300
t2_attente = 30

for i, N_t in enumerate(N_steps_list):
    
    # 💡 L'avantage de cette structure : Si vous vouliez boucler 
    # sur Lambda_SOC au lieu de N_steps_list, vous feriez simplement :
    # params_physiques['Lambda_SOC'] = nouvelle_valeur_de_la_boucle
    
    time_delays = [N_t, t2_attente, N_t]
    scan_id = [0, 2]
    simuler_et_sauvegarder(
        i=i, 
        N_t=N_t, 
        t2_attente=t2_attente, 
        phys=params_physiques,      # On passe le dict des constantes
        plot_options=params_plot,
        plot= 0, # On passe le dict des figures
        dossier_1="results_spin-orbite/Scans_rephasing_t2_50fs",
        dossier_2= "results_spin-orbite/Scans_nonrephasing1_t2_50fs"

    )

print("Toutes les simulations sont terminées !")


--- [0] Préparation du système (N_t=10, t2_attente=30) ---
system initialized
Calcul de la dynamique quantique en cours...
First scan done, starting second scan. Remaining time = First Scan Time x number of steps in second scan/number of processors
second scan done
First scan done, starting second scan. Remaining time = First Scan Time x number of steps in second scan/number of processors
second scan done
First scan done, starting second scan. Remaining time = First Scan Time x number of steps in second scan/number of processors
second scan done
First scan done, starting second scan. Remaining time = First Scan Time x number of steps in second scan/number of processors
second scan done
Enregistré : results_spin-orbite/Scans_rephasing_t2_50fs\rephasing_0000.png
Enregistré : results_spin-orbite/Scans_nonrephasing1_t2_50fs\nonrephasing_0000.png

--- [1] Préparation du système (N_t=15, t2_attente=30) ---
system initialized
Calcul de la dynamique quantique en cours...
First scan done, star

## Main code

- We can choose our parameter here, 